In [1]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.documents import Document
import ollama

# 1. Initialize the Embedding Model (Turns text into numbers)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Create some sample "Notes" (We'll use some Advanced Computer Networks concepts)
sample_notes = [
    Document(page_content="TCP (Transmission Control Protocol) is connection-oriented and guarantees delivery of packets. It uses a 3-way handshake."),
    Document(page_content="UDP (User Datagram Protocol) is connectionless and does not guarantee delivery. It is faster but less reliable, used for streaming."),
    Document(page_content="OSPF (Open Shortest Path First) is a routing protocol that uses a link-state routing algorithm.")
]

# 3. Build the Vector Database (ChromaDB)
# We are storing this in memory for Day 1. Later we will save it to disk.
print("Building the database... (This might take a few seconds)")
vectorstore = Chroma.from_documents(
    documents=sample_notes,
    embedding=embeddings
)
print("Database built successfully! 🚀")

# 4. The Retrieval & Generation Function
def ask_my_notes(question):
    print(f"\nSearching notes for: '{question}'...")
    
    # A. Search the database for the top 2 most relevant chunks of text
    results = vectorstore.similarity_search(question, k=2)
    
    # B. Combine those chunks into a single context string
    context = "\n".join([doc.page_content for doc in results])
    print(f"Found relevant context: {context}\n")
    
    # C. Build the prompt for Llama 3.2
    prompt = f"""
    You are an expert teaching assistant. Answer the student's question using ONLY the provided context. 
    If the answer is not in the context, say "I don't have enough information in your notes to answer that."
    
    Context: {context}
    
    Question: {question}
    """
    
    # D. Ask Ollama
    response = ollama.chat(model='llama3.2', messages=[
        {'role': 'user', 'content': prompt}
    ])
    
    return response['message']['content']

C:\Users\krishna saxena\AppData\Local\Temp\ipykernel_23084\1383751019.py:7: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


Building the database... (This might take a few seconds)
Database built successfully! 🚀


In [2]:
# Test 1: Ask something that IS in the notes
answer_1 = ask_my_notes("What is the difference between TCP and UDP?")
print(f"AI Answer:\n{answer_1}")

print("-" * 50)

# Test 2: Ask something that IS NOT in the notes to prove it doesn't hallucinate
answer_2 = ask_my_notes("Explain the history of the Roman Empire.")
print(f"AI Answer:\n{answer_2}")


Searching notes for: 'What is the difference between TCP and UDP?'...
Found relevant context: UDP (User Datagram Protocol) is connectionless and does not guarantee delivery. It is faster but less reliable, used for streaming.
TCP (Transmission Control Protocol) is connection-oriented and guarantees delivery of packets. It uses a 3-way handshake.

AI Answer:
The main difference between TCP and UDP is that TCP is connection-oriented, which means it establishes a connection before sending data, and guarantees delivery of packets. On the other hand, UDP is connectionless, which means it does not establish a connection before sending data, and does not guarantee delivery of packets.
--------------------------------------------------

Searching notes for: 'Explain the history of the Roman Empire.'...
Found relevant context: OSPF (Open Shortest Path First) is a routing protocol that uses a link-state routing algorithm.
TCP (Transmission Control Protocol) is connection-oriented and guarantees

In [4]:
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter # Updated import
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings # Updated import
import ollama

# 1. THE EYES: Scan the image
print("👀 Scanning your notes...")
converter = DocumentConverter()
# Make sure this filename matches your image!
result = converter.convert("WhatsApp Image 2026-03-09 at 2.53.29 PM.jpeg") 
markdown_text = result.document.export_to_markdown()

# 2. THE CHUNKER: Now using the specific text_splitter package
print("✂️ Splitting text into chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = [Document(page_content=markdown_text)]
chunks = text_splitter.split_documents(docs)

# 3. THE MEMORY
print("🧠 Building the brain...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

# 4. THE MOUTH
question = "What is the main topic of these notes?" 
print(f"\n🔍 Searching for: '{question}'")

results = vectorstore.similarity_search(question, k=2)
context = "\n".join([doc.page_content for doc in results])

response = ollama.chat(model='llama3.2', messages=[
    {'role': 'user', 'content': f"Context: {context}\n\nQuestion: {question}"}
])

print(f"\n🤖 AI Answer:\n{response['message']['content']}")

[INFO] 2026-03-09 15:28:02,338 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 15:28:02,348 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 15:28:02,349 [RapidOCR] main.py:53: Using C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx


👀 Scanning your notes...


[INFO] 2026-03-09 15:28:02,687 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 15:28:02,692 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 15:28:02,694 [RapidOCR] main.py:53: Using C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 15:28:02,857 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 15:28:02,881 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-03-09 15:28:02,882 [RapidOCR] main.py:53: Using C:\Users\krishna saxena\smart-notes-explainer\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx


✂️ Splitting text into chunks...
🧠 Building the brain...

🔍 Searching for: 'What is the main topic of these notes?'

🤖 AI Answer:
The main topic of these notes appears to be a list of permissions or access levels for a special app, likely related to Android device settings.
